In [2]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

# Here comes the data evaluation

In [2]:
# import from pickle

HOME_DIR = "/Users/oliver/Documents/cryolab/p5control-bluefors-evaluation"
sys.path.append(HOME_DIR)

from utilities.ivplot import IVPlot

importlib.reload(sys.modules["utilities.ivplot"])

eva = IVPlot()
eva.title = f"amplitude at 18.3GHz"
eva.sub_folder = (
    "/Users/oliver/Documents/cryolab/superconductivity/evaluation/TB irradiation/data/"
)
eva.loadData()

Vbias_mV = eva.mapped["voltage_axis"] * 1e3
Ibias_nA = eva.mapped["current_axis"] * 1e9
Aout_mV = eva.mapped["y_axis"] * 1e3
dGexp_G0 = eva.up_sweep["differential_conductance"]
dRexp_R0 = eva.up_sweep["differential_resistance"] * sc.G0_muS / 1e6
Iexp_nA = eva.up_sweep["current"] * 1e9
nu_GHz = 18.3

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) ... BaseClass initialized.
(base plot) ... BasePlot initialized.
(iv plot) ... IVPlot initialized.
(base) amplitude at 18.3GHz
(base) loadData()


In [3]:
np.savez_compressed(
    "amp_18.3GHz_eva.npz",
    Vbias_mV=Vbias_mV,
    Ibias_nA=Ibias_nA,
    Aout_mV=Aout_mV,
    dGexp_G0=dGexp_G0,
    dRexp_R0=dRexp_R0,
    Iexp_nA=Iexp_nA,
    nu_GHz=nu_GHz,
)

In [4]:
# import from pickle

HOME_DIR = "/Users/oliver/Documents/cryolab/p5control-bluefors-evaluation"
sys.path.append(HOME_DIR)

from utilities.ivplot import IVPlot

importlib.reload(sys.modules["utilities.ivplot"])

eva = IVPlot()
eva.title = f"amplitude at 13.6GHz"
eva.sub_folder = (
    "/Users/oliver/Documents/cryolab/superconductivity/evaluation/TB irradiation/data/"
)
eva.loadData()

Vbias_mV = eva.mapped["voltage_axis"] * 1e3
Ibias_nA = eva.mapped["current_axis"] * 1e9
Aout_mV = eva.mapped["y_axis"] * 1e3
dGexp_G0 = eva.up_sweep["differential_conductance"]
dRexp_R0 = eva.up_sweep["differential_resistance"] * sc.G0_muS / 1e6
Iexp_nA = eva.up_sweep["current"] * 1e9
nu_GHz = 13.6

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) ... BaseClass initialized.
(base plot) ... BasePlot initialized.
(iv plot) ... IVPlot initialized.
(base) amplitude at 13.6GHz
(base) loadData()


In [5]:
np.savez_compressed(
    "amp_13.6GHz_eva.npz",
    Vbias_mV=Vbias_mV,
    Ibias_nA=Ibias_nA,
    Aout_mV=Aout_mV,
    dGexp_G0=dGexp_G0,
    dRexp_R0=dRexp_R0,
    Iexp_nA=Iexp_nA,
    nu_GHz=nu_GHz,
)

# Temperature Study

In [1]:
# generate TB cache
# from superconductivity.utilities import make_cache, save_cache

# cache = make_cache(name="TB-temperature")
# save_cache(cache)

In [3]:
# load cache
from superconductivity.utilities import load_cache, save_cache

cache = load_cache("TB-temperature")

In [4]:
# get data
from superconductivity.evaluation import FileSpec

cache.filespec = FileSpec(
    h5path="OI-25c-09 2025-04-15 unbroken 1.hdf5",
    location="/Users/oliver/Documents/measurement data/25 04 OI-25c-09",
    measurement="temperatures",
)
cache.mkeys = cache.filespec.mkeys()
cache.skeys = cache.filespec.skeys()
_ = save_cache(cache)

In [5]:
# get keys
from superconductivity.evaluation import KeysSpec
from superconductivity.evaluation import get_keys

cache.keysspec = KeysSpec(
    strip0="heater_",
    strip1="muW",
    remove_key="no_heater",
    add_key=[
        ("no_heater", 0.0),
    ],
    limits=(None, None),
    norm=1e-6,
    label="Aout_mV",
)

cache.skeys = get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = save_cache(cache)

In [6]:
# get traces
from superconductivity.evaluation import TraceSpec
from superconductivity.evaluation import get_traces

cache.tracespec = TraceSpec(
    AmpV=1000,
    AmpI=1000,
    trigger_values=1,
    skip_edges=5,
)

cache.traces = get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = save_cache(cache)

In [ ]:
# offset analysis
from superconductivity.evaluation import OffsetSpec
from superconductivity.evaluation import offset_analysis

cache.offsetspec = OffsetSpec(
    Vbins_mV=np.linspace(-0.5, 0.5, 51),
    Ibins_nA=np.linspace(-5.0, 5.0, 51),
    Voffscan_mV=np.linspace(-0.045, 0.045, 451),
    Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
    cutoff_Hz=43.7,
    sampling_Hz=137.0,
)

cache.offsetanalysis = offset_analysis(
    traces=cache.traces,
    spec=cache.offsetspec,
)
_ = save_cache(cache)

offset_analysis:   0%|          | 0/151 [00:00<?, ?trace/s]

In [ ]:
# sampling
from superconductivity.evaluation import SamplingSpec
from superconductivity.evaluation import sample

cache.samplingspec = SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 1601),
    Ibins_nA=np.linspace(-30.0, 30.0, 2001),
    Voff_mV=cache.offsetanalysis.Voff_mV,
    Ioff_nA=cache.offsetanalysis.Ioff_nA,
    cutoff_Hz=43.7,
    sampling_Hz=137.0,
    median_bins=3,
    sigma_bins=2.0,
)

cache.samplingspec.keys()

cache.exp_v, cache.exp_i = sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = save_cache(cache)

In [ ]:
# get temperature
from utilities.ivplot import IVPlot

V_exp_mV = np.linspace(-0.1, 0.8, 451, dtype="float64")
V_mV = np.linspace(0, 0.7, 351, dtype="float64")

V_mV = np.where(V_mV == 0, np.nan, V_mV)
V_mask = np.logical_not(np.isnan(V_mV))
VV_mV = np.concatenate((V_mV, V_mV), dtype="float64")

eva = IVPlot()
eva.file_directory = "/Users/oliver/Documents/measurement data/"
eva.file_folder = "25 04 OI-25c-09/"
eva.file_name = "OI-25c-09 2025-04-15 unbroken 1.hdf5"
eva.title = "fitting study"
eva.sub_folder = ""

eva.setAmplifications(1000, 1000)
eva.setV(
    voltage_minimum=-1.0,
    voltage_maximum=1.0,
    voltage_bins=21,
)
eva.setI(current_minimum=-0.1e-8, current_maximum=0.8e-8, current_bins=9)
eva.setT(0.2, 1.4, 240)
eva.downsample_frequency = 137

# eva.showMeasurements()

eva.setMeasurement("temperatures")
eva.showKeys()
eva.setKeys(index_0=7, index_1=-3, norm=1e-6, to_pop="no_heater")
(eva.up_sweep,) = eva.getMaps([1])

(eva.up_sweep,) = eva.getMapsTemperature([eva.up_sweep])
eva.y_axis = eva.temperature_axis

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) ... BaseClass initialized.
(base plot) ... BasePlot initialized.
(iv plot) ... IVPlot initialized.
(base) fitting study
(base eva) Measurement keys preview: ['heater_00000.013muW', 'heater_00000.070muW', 'heater_03135.622muW', 'no_heater']
/Users/oliver/Documents/cryolab/p5control-bluefors-evaluation/utilities/basefunctions.py:56: RankWarning: Polyfit may be poorly conditioned
  poly = np.polyfit(xx[not_nans], nu_x[not_nans], 1)


TypeError: only 0-dimensional arrays can be converted to Python scalars

In [20]:
# import from pickle

HOME_DIR = "/Users/oliver/Documents/cryolab/p5control-bluefors-evaluation"
sys.path.append(HOME_DIR)

from utilities.ivplot import IVPlot

importlib.reload(sys.modules["utilities.ivplot"])

eva = IVPlot()
eva.title = f"temperature study"
eva.sub_folder = (
    "/Users/oliver/Documents/cryolab/superconductivity/evaluation/TB temperature/data/"
)
eva.loadData()

Vbias_mV = eva.mapped["voltage_axis"] * 1e3
Ibias_nA = eva.mapped["current_axis"] * 1e9
Tbath_K = eva.mapped["y_axis"]
dGexp_G0 = eva.up_sweep["differential_conductance"]
dRexp_R0 = eva.up_sweep["differential_resistance"] * sc.G0_muS / 1e6
Iexp_nA = eva.up_sweep["current"] * 1e9

(base) ... BaseClass initialized.
(base eva) ... BaseEvaluation initialized.
(iv eva) ... IVEvaluation initialized.
(base) ... BaseClass initialized.
(base plot) ... BasePlot initialized.
(iv plot) ... IVPlot initialized.
(base) temperature study
(base) loadData()


In [21]:
Tbath_K

array([0.2  , 0.205, 0.21 , 0.215, 0.22 , 0.225, 0.23 , 0.235, 0.24 ,
       0.245, 0.25 , 0.255, 0.26 , 0.265, 0.27 , 0.275, 0.28 , 0.285,
       0.29 , 0.295, 0.3  , 0.305, 0.31 , 0.315, 0.32 , 0.325, 0.33 ,
       0.335, 0.34 , 0.345, 0.35 , 0.355, 0.36 , 0.365, 0.37 , 0.375,
       0.38 , 0.385, 0.39 , 0.395, 0.4  , 0.405, 0.41 , 0.415, 0.42 ,
       0.425, 0.43 , 0.435, 0.44 , 0.445, 0.45 , 0.455, 0.46 , 0.465,
       0.47 , 0.475, 0.48 , 0.485, 0.49 , 0.495, 0.5  , 0.505, 0.51 ,
       0.515, 0.52 , 0.525, 0.53 , 0.535, 0.54 , 0.545, 0.55 , 0.555,
       0.56 , 0.565, 0.57 , 0.575, 0.58 , 0.585, 0.59 , 0.595, 0.6  ,
       0.605, 0.61 , 0.615, 0.62 , 0.625, 0.63 , 0.635, 0.64 , 0.645,
       0.65 , 0.655, 0.66 , 0.665, 0.67 , 0.675, 0.68 , 0.685, 0.69 ,
       0.695, 0.7  , 0.705, 0.71 , 0.715, 0.72 , 0.725, 0.73 , 0.735,
       0.74 , 0.745, 0.75 , 0.755, 0.76 , 0.765, 0.77 , 0.775, 0.78 ,
       0.785, 0.79 , 0.795, 0.8  , 0.805, 0.81 , 0.815, 0.82 , 0.825,
       0.83 , 0.835,